# Tabela dinâmica

Uma tabela dinâmica (pivot table) é uma forma de resumir e reorganizar dados de um DataFrame com base em um objetivo específico. Ela utiliza funções de agregação (como soma, média, contagem) para consolidar grandes volumes de dados em uma visão mais simples.

Em uma pivot table:

- as linhas representam uma variável de interesse,
- as colunas representam outra variável,
- e as células mostram os valores agregados resultantes da combinação entre essas variáveis.

Além disso, é comum incluir totais marginais, que representam os totais por linha e por coluna, facilitando ainda mais a análise.

Esse tipo de tabela permite visualizar rapidamente padrões, comparações e relações entre variáveis, tornando a análise de dados mais intuitiva.

A pivot_table() responde perguntas do tipo:
- Como uma variável se comporta em relação a duas outras?
- Quero ver a média (ou soma, etc) organizada como uma tabela cruzando duas ou mais variáveis

In [89]:
# Importando as bibliotecas 
import pandas as pd
import numpy as np

In [90]:
# Vamos ler um arquivo csv
df = pd.read_csv('datasets/cwurData.csv')
df.head()

,world_rank,institution,country,national_rank,quality_of_education,alumni_employment,quality_of_faculty,publications,influence,citations,broad_impact,patents,score,year
0,1,Harvard University,USA,1,7,9,1,1,1,1,NaN,5,100.00,2012
1,2,Massachusetts Institute of Technology,USA,2,9,17,3,12,4,4,NaN,1,91.67,2012
2,3,Stanford University,USA,3,17,11,5,4,2,2,NaN,15,89.50,2012
3,4,University of Cambridge,United Kingdom,1,10,24,4,16,16,11,NaN,50,86.17,2012
4,5,California Institute of Technology,USA,4,2,29,7,37,22,22,NaN,18,85.21,2012


Podemos ver as colunas de classificação das intituições (world_rank), país (country), qualidade da educação (quality_of_education) e outras métricas, além da pontuação geral. 

In [91]:
# Vamos supor que queremos criar uma nova coluna chamada 'Rank_Level', em que teremos as intituições cujo valor de world_rank esteja entre a posição 1 e a 100 como o primeiro nível, entre a posição 101 a 200 são o segundo nível, e entre o ranking 201 a 300 são o terceiro nível. Dessa forma, depois de 301, vamos apenas classificar como outro nível de classificação das universidades. 

# Para isso, vamos criar uma Série de dado categórico
minha_categoria = pd.cut(df['world_rank'], bins = [0, 100, 200, 300, np.inf], 
                         labels = ['nível 1', 'nível 2', 'nível 3', 'nível 4'])
# OBS: np.inf representa infinito

# Vamos inverter a ordem, pois está sendo considerado nível 4 > nível 3
minha_categoria = minha_categoria.cat.reorder_categories(
    ['nível 4', 'nível 3', 'nível 2', 'nível 1'],
    ordered = True
)
# OBS: Usamos .cat é porque é um dado categórico 

# Vamos usar agora
df['rank_level'] = minha_categoria

df.head()


,world_rank,institution,country,national_rank,quality_of_education,alumni_employment,quality_of_faculty,publications,influence,citations,broad_impact,patents,score,year,rank_level
0,1,Harvard University,USA,1,7,9,1,1,1,1,NaN,5,100.00,2012,nível 1
1,2,Massachusetts Institute of Technology,USA,2,9,17,3,12,4,4,NaN,1,91.67,2012,nível 1
2,3,Stanford University,USA,3,17,11,5,4,2,2,NaN,15,89.50,2012,nível 1
3,4,University of Cambridge,United Kingdom,1,10,24,4,16,16,11,NaN,50,86.17,2012,nível 1
4,5,California Institute of Technology,USA,4,2,29,7,37,22,22,NaN,18,85.21,2012,nível 1


In [92]:
# Através da função pivot_table() iremos criar colunas de valores resumidos com novas colunas de comparação contendo dados processados. 
# Desejamos comparar a coluna 'rank_level' por país, por 'country', onde se localiza a universidade e, então, queremos compará-la em termos de pontuação geral. 

# Para isso, diremos ao Pandas que queremos os valores que estão em 'score', e o indexador será 'country', e iremos obter os níveis do rank_level. 

df.pivot_table(index = 'country', columns = 'rank_level', values = 'score',
               aggfunc = [np.mean]).head()

mean                             
rank_level    nível 4    nível 3  nível 2  nível 1
country                                           
Argentina   44.672857        NaN      NaN      NaN
Australia   44.645750  47.285000  49.2425  47.9425
Austria     44.864286  47.066667      NaN      NaN
Belgium     45.081000  46.746667  49.0840  51.8750
Brazil      44.499706        NaN  49.5650      NaN

Percebe-se que alguns valores não são numéricos, o que significa que pode não ter esse nível de faculdade em determinado país.

OBS: O parâmetro 'aggfunc' pode receber funções criadas pelo programador.

In [93]:
# As tabelas dinâmicas não são limitadas a uma função que você queira aplicar. Você pode passar um uma lista das diferentes funções a serem aplicadas no parâmetro aggfunc, e o pandas lhe fornecerá o resultado usando nomes de colunas hierárquicas. 
# Vamos tentar aquela mesma consulta, mas passando a função 'max()' também.

df.pivot_table(index = 'country', columns = 'rank_level', values = 'score',
                aggfunc = [np.mean, np.max]).head()

mean                                  max                  \
rank_level    nível 4    nível 3  nível 2  nível 1 nível 4 nível 3 nível 2   
country                                                                      
Argentina   44.672857        NaN      NaN      NaN   45.66     NaN     NaN   
Australia   44.645750  47.285000  49.2425  47.9425   45.97   47.47   50.40   
Austria     44.864286  47.066667      NaN      NaN   46.29   47.78     NaN   
Belgium     45.081000  46.746667  49.0840  51.8750   46.21   47.14   49.73   
Brazil      44.499706        NaN  49.5650      NaN   46.08     NaN   49.82   

                    
rank_level nível 1  
country             
Argentina      NaN  
Australia    51.61  
Austria        NaN  
Belgium      52.03  
Brazil         NaN

Perceba que temos tanto mean quanto max na tabela. 

In [94]:
# Também podemos resumir os valores em um determinado grupo de nível superior. Por exemplo, se quisermos ver a média geral do país com 'mean' e quisermos ver o 'max' do 'max', podemos indicar que queremos que o pandas faça isso fornecendo valores marginais. 

df.pivot_table(index = 'country', columns = 'rank_level', values = 'score',
                aggfunc = [np.mean, np.max], margins = True).head()

mean                                             max          \
rank_level    nível 4    nível 3  nível 2  nível 1        All nível 4 nível 3   
country                                                                         
Argentina   44.672857        NaN      NaN      NaN  44.672857   45.66     NaN   
Australia   44.645750  47.285000  49.2425  47.9425  45.825517   45.97   47.47   
Austria     44.864286  47.066667      NaN      NaN  45.139583   46.29   47.78   
Belgium     45.081000  46.746667  49.0840  51.8750  47.011000   46.21   47.14   
Brazil      44.499706        NaN  49.5650      NaN  44.781111   46.08     NaN   

                                   
rank_level nível 2 nível 1    All  
country                            
Argentina      NaN     NaN  45.66  
Australia    50.40   51.61  51.61  
Austria        NaN     NaN  47.78  
Belgium      49.73   52.03  52.03  
Brazil       49.82     NaN  49.82

Quando utilizamos margins=True, estamos solicitando que a tabela inclua totais gerais, calculados a partir dos dados originais, e não a partir dos valores já agregados na tabela.

Esses totais seguem exatamente as funções definidas em aggfunc. No caso de np.mean e np.max, isso significa que:

Para cada país, além dos valores por nível (rank_level), teremos um valor geral (All) que representa:
- a média de todas as observações daquele país, considerando todas as universidades (e não a média das médias por nível);
- o valor máximo entre todas as observações daquele país, e não o máximo entre os resultados agregados de cada nível.

Além disso, a linha e coluna “All” representam os totais globais, ou seja, os resultados das funções de agregação aplicadas a todo o conjunto de dados.

OBS: Uma pivot table é um DataFrame multinível. 

In [95]:
# Como uma pivot_table é um DataFrame, podemos acessar os dados de forma parecida como num DataFrame normal. 

novo_df = df.pivot_table(index = 'country', columns = 'rank_level', values = 'score',
                aggfunc = [np.mean, np.max], margins = True)

# Vamos ver os índices
print(novo_df.index)

# Vamos ver as colunas
print(novo_df.columns)

Index(['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Bulgaria',
       'Canada', 'Chile', 'China', 'Colombia', 'Croatia', 'Cyprus',
       'Czech Republic', 'Denmark', 'Egypt', 'Estonia', 'Finland', 'France',
       'Germany', 'Greece', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Iran',
       'Ireland', 'Israel', 'Italy', 'Japan', 'Lebanon', 'Lithuania',
       'Malaysia', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Poland',
       'Portugal', 'Puerto Rico', 'Romania', 'Russia', 'Saudi Arabia',
       'Serbia', 'Singapore', 'Slovak Republic', 'Slovenia', 'South Africa',
       'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Taiwan', 'Thailand',
       'Turkey', 'USA', 'Uganda', 'United Arab Emirates', 'United Kingdom',
       'Uruguay', 'All'],
      dtype='str', name='country')
MultiIndex([('mean', 'nível 4'),
            ('mean', 'nível 3'),
            ('mean', 'nível 2'),
            ('mean', 'nível 1'),
            ('mean',     'All'),
            ( 'max', 

Percebe-se que as colunas são hierárquicas. Os índices das colunas de nível superior tem duas categorias, 'mean' e 'max', e os índices das coolunas de nível inferior têm quatro categorias, que são os quatro níveis do ranking.

In [96]:
# Vamos consultar a pivot table

# Como consultar se quisermos obter as pontuações médias das primeiras universidades de níveis superiores de cada país? 

novo_df.loc[:, ('mean', 'nível 1')].head()


country
Argentina        NaN
Australia    47.9425
Austria          NaN
Belgium      51.8750
Brazil           NaN
Name: (mean, nível 1), dtype: float64

Perceba que as colunas tem que ser passadas em forma de tupla, respeitando a hierarquia. No exemplo acima, 'mean' é superior à 'nível 1'.

In [97]:
# Outra forma tão boa quanto
novo_df[('mean', 'nível 1')].head()

country
Argentina        NaN
Australia    47.9425
Austria          NaN
Belgium      51.8750
Brazil           NaN
Name: (mean, nível 1), dtype: float64

ATENÇÃO: Usa-se tupla pois as colunas são organizadas como um MultiIndex (hierarquia). Assim, é necessário especificar todos os níveis da coluna para acessá-la de forma não ambígua. A lista pode mudar a ordem, a tupla não. 

In [98]:
# Forma computacionalmente mais custosa:
novo_df['mean']['nível 1'].head()

country
Argentina        NaN
Australia    47.9425
Austria          NaN
Belgium      51.8750
Brazil           NaN
Name: nível 1, dtype: float64

##### E se quisermos mais de um valor de nível superior e/ou inferior?

In [99]:
novo_df.loc[:, [('mean', 'nível 1'), ('max', 'nível 1')]].head()

,mean,max
rank_level,nível 1,nível 1
country,,
Argentina,NaN,NaN
Australia,47.9425,51.61
Austria,NaN,NaN
Belgium,51.8750,52.03
Brazil,NaN,NaN


Esse jeito é meio ruim, então talvez compense perder eficiencia computacional e utilizar o jeito de selecionar colunas separadas, já que o pandas não entendo o produto cartesiano nesse caso
- novo_df[níveis superiores][níveis inferiores]

In [100]:
# Ou utilizar 
idx = pd.IndexSlice

novo_df.loc[:, idx[['mean', 'max'], ['nível 1', 'nível 2']]].head()

mean              max        
rank_level  nível 1  nível 2 nível 1 nível 2
country                                     
Argentina       NaN      NaN     NaN     NaN
Australia   47.9425  49.2425   51.61   50.40
Austria         NaN      NaN     NaN     NaN
Belgium     51.8750  49.0840   52.03   49.73
Brazil          NaN  49.5650     NaN   49.82

##### E se quisermos encontrar o país com a maior pontuação média do nível 1?

In [101]:
novo_df[('mean', 'nível 1')].idxmax()

'United Kingdom'

##### Podemos ver formas diferentes da pivot table

Se quisermos alcançar uma forma diferente, podemos usar as funções:
- stack()
- unstack()

Empilhar (stack) é pivotar o índice da coluna mais baixa para se tornar o índice de linha mais interno. <br>
Desempilhar (unstack) é o contrário, pivotando o índice da linha mais interna para se tornar o mais baixo índice da coluna. 

Basicamente: 
- stack() transforma colunas em linhas, deixando o DataFrame mais vertical.
- unstack() transforma linhas em colunas, deixando o DataFrame mais horizontal.

In [102]:
# Vamos ver como o dataframe é originalmente. 
novo_df.head()

mean                                             max          \
rank_level    nível 4    nível 3  nível 2  nível 1        All nível 4 nível 3   
country                                                                         
Argentina   44.672857        NaN      NaN      NaN  44.672857   45.66     NaN   
Australia   44.645750  47.285000  49.2425  47.9425  45.825517   45.97   47.47   
Austria     44.864286  47.066667      NaN      NaN  45.139583   46.29   47.78   
Belgium     45.081000  46.746667  49.0840  51.8750  47.011000   46.21   47.14   
Brazil      44.499706        NaN  49.5650      NaN  44.781111   46.08     NaN   

                                   
rank_level nível 2 nível 1    All  
country                            
Argentina      NaN     NaN  45.66  
Australia    50.40   51.61  51.61  
Austria        NaN     NaN  47.78  
Belgium      49.73   52.03  52.03  
Brazil       49.82     NaN  49.82

In [103]:
# Vamos a um exemplo:

novo_df = novo_df.stack()
novo_df.head()


mean    max
country   rank_level                  
Argentina nível 4     44.672857  45.66
          nível 3           NaN    NaN
          nível 2           NaN    NaN
          nível 1           NaN    NaN
          All         44.672857  45.66

Podemos observar que as colunas foram transpostas para linhas. 

In [104]:
# Vamos desempilhar
novo_df.unstack().head()

mean                                             max          \
rank_level    nível 4    nível 3  nível 2  nível 1        All nível 4 nível 3   
country                                                                         
Argentina   44.672857        NaN      NaN      NaN  44.672857   45.66     NaN   
Australia   44.645750  47.285000  49.2425  47.9425  45.825517   45.97   47.47   
Austria     44.864286  47.066667      NaN      NaN  45.139583   46.29   47.78   
Belgium     45.081000  46.746667  49.0840  51.8750  47.011000   46.21   47.14   
Brazil      44.499706        NaN  49.5650      NaN  44.781111   46.08     NaN   

                                   
rank_level nível 2 nível 1    All  
country                            
Argentina      NaN     NaN  45.66  
Australia    50.40   51.61  51.61  
Austria        NaN     NaN  47.78  
Belgium      49.73   52.03  52.03  
Brazil       49.82     NaN  49.82

In [105]:
# E se desempilharmos duas vezes seguidas? (Seria como desempilhar uma vez o original)
novo_df.unstack().unstack().head()

      rank_level  country  
mean  nível 4     Argentina    44.672857
                  Australia    44.645750
                  Austria      44.864286
                  Belgium      45.081000
                  Brazil       44.499706
dtype: float64

Acabamos desempilhando tudo para formar uma só coluna, o que nos dá um tipo Série